# Advanced Problems with Solutions: Tuples as Data Structures

This notebook contains advanced practice problems on Python tuples, including:

- tuple packing and unpacking
- tuple immutability vs mutable contained objects
- tuples as lightweight data records
- homogeneous collections of heterogeneous records
- extended unpacking
- tuple-based return values
- `enumerate`, sorting, grouping, validation, and design trade-offs

Each problem is followed by a complete solution and basic tests.

## Best-Practice Rules for These Problems

1. Prefer readable unpacking over hard-coded numeric indexes when the tuple structure is central to the task.
2. Use `_` or `*_` for intentionally ignored values.
3. Treat tuple records as immutable contracts: do not silently accept malformed records.
4. Remember that a tuple is immutable, but objects inside it may still be mutable.
5. Use generator expressions when you only need to aggregate values once.
6. Keep record-processing functions pure when possible: return new tuples instead of mutating input data.

## Problem 1 — Validate and Normalize City Records

You are given city records as tuples in this format:

```python
(name, country, population)
```

Write a function `normalize_cities(records)` that:

- accepts any iterable of records
- verifies that each record has exactly 3 fields
- verifies that `name` and `country` are non-empty strings
- verifies that `population` is a non-negative integer
- returns a tuple of normalized records
- normalizes city and country names by stripping whitespace
- raises `ValueError` with a helpful message for invalid records

Return value example:

```python
(("London", "UK", 8780000), ("New York", "USA", 8500000))
```

In [1]:
def normalize_cities(records):
    normalized = []

    for index, record in enumerate(records):
        try:
            name, country, population = record
        except ValueError as exc:
            raise ValueError(
                f"Record #{index} must contain exactly 3 fields: "
                "(name, country, population)"
            ) from exc

        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"Record #{index} has an invalid city name: {name!r}")

        if not isinstance(country, str) or not country.strip():
            raise ValueError(f"Record #{index} has an invalid country: {country!r}")

        if not isinstance(population, int) or population < 0:
            raise ValueError(
                f"Record #{index} has an invalid population: {population!r}"
            )

        normalized.append((name.strip(), country.strip(), population))

    return tuple(normalized)


# Tests
raw_cities = [
    (" London ", " UK ", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000),
]

expected = (
    ("London", "UK", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000),
)

assert normalize_cities(raw_cities) == expected
assert isinstance(normalize_cities(raw_cities), tuple)

try:
    normalize_cities([("Bad City", "Nowhere", -1)])
except ValueError as ex:
    assert "population" in str(ex)
else:
    raise AssertionError("Expected ValueError for negative population")

normalize_cities(raw_cities)

(('London', 'UK', 8780000),
 ('New York', 'USA', 8500000),
 ('Beijing', 'China', 21000000))

## Problem 2 — Population Analytics with Tuple Records

Given normalized city records:

```python
(name, country, population)
```

Write `population_report(cities)` that returns a tuple:

```python
(total_population, average_population, largest_city_record, city_names)
```

Where:

- `total_population` is an integer
- `average_population` is a float
- `largest_city_record` is the full tuple record for the city with the largest population
- `city_names` is a tuple containing only city names in original order

The function should raise `ValueError` if `cities` is empty.

In [2]:
def population_report(cities):
    cities = tuple(cities)

    if not cities:
        raise ValueError("cities cannot be empty")

    total_population = sum(population for _, _, population in cities)
    average_population = total_population / len(cities)
    largest_city_record = max(cities, key=lambda city: city[2])
    city_names = tuple(name for name, _, _ in cities)

    return total_population, average_population, largest_city_record, city_names


# Tests
cities = (
    ("London", "UK", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000),
)

report = population_report(cities)

assert report[0] == 38_280_000
assert report[1] == 12_760_000.0
assert report[2] == ("Beijing", "China", 21_000_000)
assert report[3] == ("London", "New York", "Beijing")

try:
    population_report(())
except ValueError as ex:
    assert "empty" in str(ex)
else:
    raise AssertionError("Expected ValueError for empty input")

report

(38280000,
 12760000.0,
 ('Beijing', 'China', 21000000),
 ('London', 'New York', 'Beijing'))

## Problem 3 — Extended Unpacking for Market Records

A market record is represented as:

```python
(symbol, year, month, day, open_price, high_price, low_price, close_price)
```

Write `summarize_market_record(record)` that uses extended unpacking to return:

```python
(symbol, date_tuple, close_price, intraday_range)
```

Where:

```python
date_tuple == (year, month, day)
intraday_range == high_price - low_price
```

The implementation must avoid direct numeric indexing like `record[7]`.

In [3]:
def summarize_market_record(record):
    symbol, year, month, day, open_price, high_price, low_price, close_price = record
    date_tuple = year, month, day
    intraday_range = high_price - low_price
    return symbol, date_tuple, close_price, intraday_range


def summarize_market_record_extended(record):
    symbol, year, month, day, *prices = record
    open_price, high_price, low_price, close_price = prices
    return symbol, (year, month, day), close_price, high_price - low_price


# Tests
record = ("DJIA", 2018, 1, 19, 25_987, 26_072, 25_942, 26_072)

expected = ("DJIA", (2018, 1, 19), 26_072, 130)

assert summarize_market_record(record) == expected
assert summarize_market_record_extended(record) == expected

summarize_market_record_extended(record)

('DJIA', (2018, 1, 19), 26072, 130)

## Problem 4 — Safe Tuple Update Without Mutation

A tuple is immutable, so updating a tuple record means creating a new tuple.

A user profile is represented as:

```python
(username, email, is_active, roles)
```

Where `roles` is itself a tuple of strings.

Write `add_role(profile, role)` that:

- returns a new profile tuple
- appends `role` to the roles tuple
- does not duplicate an existing role
- does not mutate the original tuple
- raises `ValueError` for an empty role

In [4]:
def add_role(profile, role):
    username, email, is_active, roles = profile

    if not isinstance(role, str) or not role.strip():
        raise ValueError("role must be a non-empty string")

    role = role.strip()

    if role in roles:
        return profile

    return username, email, is_active, roles + (role,)


# Tests
profile = ("ada", "ada@example.com", True, ("reader", "editor"))

updated = add_role(profile, "admin")

assert profile == ("ada", "ada@example.com", True, ("reader", "editor"))
assert updated == ("ada", "ada@example.com", True, ("reader", "editor", "admin"))
assert updated is not profile
assert add_role(updated, "admin") == updated

try:
    add_role(profile, "   ")
except ValueError as ex:
    assert "role" in str(ex)
else:
    raise AssertionError("Expected ValueError for empty role")

updated

('ada', 'ada@example.com', True, ('reader', 'editor', 'admin'))

## Problem 5 — Tuple Immutability vs Mutable Contained Objects

Consider a tuple containing mutable lists:

```python
inventory = (
    ("apples", [10, 12, 9]),
    ("oranges", [7, 8, 6]),
)
```

Write two functions:

1. `unsafe_add_stock(inventory, item_name, amount)`  
   Mutates the inner list for the matching item.

2. `safe_add_stock(inventory, item_name, amount)`  
   Returns a completely new tuple structure and does not mutate the original nested lists.

This problem demonstrates that tuple immutability is shallow.

In [5]:
def unsafe_add_stock(inventory, item_name, amount):
    for name, stock_history in inventory:
        if name == item_name:
            stock_history.append(amount)
            return inventory

    raise ValueError(f"Unknown item: {item_name!r}")


def safe_add_stock(inventory, item_name, amount):
    updated_records = []
    found = False

    for name, stock_history in inventory:
        if name == item_name:
            updated_records.append((name, tuple(stock_history) + (amount,)))
            found = True
        else:
            updated_records.append((name, tuple(stock_history)))

    if not found:
        raise ValueError(f"Unknown item: {item_name!r}")

    return tuple(updated_records)


# Tests
inventory = (
    ("apples", [10, 12, 9]),
    ("oranges", [7, 8, 6]),
)

unsafe_result = unsafe_add_stock(inventory, "apples", 11)
assert unsafe_result is inventory
assert inventory[0][1] == [10, 12, 9, 11]

inventory = (
    ("apples", [10, 12, 9]),
    ("oranges", [7, 8, 6]),
)

safe_result = safe_add_stock(inventory, "apples", 11)

assert inventory[0][1] == [10, 12, 9]
assert safe_result == (
    ("apples", (10, 12, 9, 11)),
    ("oranges", (7, 8, 6)),
)

safe_result

(('apples', (10, 12, 9, 11)), ('oranges', (7, 8, 6)))

## Problem 6 — Group Tuple Records by Country

Given city records:

```python
(name, country, population)
```

Write `group_by_country(cities)` that returns a tuple of country summary records:

```python
(country, city_count, total_population)
```

Requirements:

- return summaries sorted alphabetically by country
- use tuple unpacking in the loop
- do not mutate the original city records

In [6]:
def group_by_country(cities):
    totals = {}

    for name, country, population in cities:
        city_count, total_population = totals.get(country, (0, 0))
        totals[country] = city_count + 1, total_population + population

    return tuple(
        (country, city_count, total_population)
        for country, (city_count, total_population) in sorted(totals.items())
    )


# Tests
cities = (
    ("London", "UK", 8_780_000),
    ("Manchester", "UK", 553_000),
    ("New York", "USA", 8_500_000),
    ("Boston", "USA", 675_000),
    ("Beijing", "China", 21_000_000),
)

expected = (
    ("China", 1, 21_000_000),
    ("UK", 2, 9_333_000),
    ("USA", 2, 9_175_000),
)

assert group_by_country(cities) == expected

group_by_country(cities)

(('China', 1, 21000000), ('UK', 2, 9333000), ('USA', 2, 9175000))

## Problem 7 — Rank Cities with `enumerate` and Tuple Sorting

Given city records:

```python
(name, country, population)
```

Write `rank_cities(cities)` that returns records in this format:

```python
(rank, name, country, population)
```

Requirements:

- sort cities by population descending
- rank starts at 1
- preserve the original tuple records
- use `enumerate`

In [7]:
def rank_cities(cities):
    sorted_cities = sorted(cities, key=lambda city: city[2], reverse=True)

    return tuple(
        (rank, name, country, population)
        for rank, (name, country, population) in enumerate(sorted_cities, start=1)
    )


# Tests
cities = (
    ("London", "UK", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000),
)

expected = (
    (1, "Beijing", "China", 21_000_000),
    (2, "London", "UK", 8_780_000),
    (3, "New York", "USA", 8_500_000),
)

assert rank_cities(cities) == expected

rank_cities(cities)

((1, 'Beijing', 'China', 21000000),
 (2, 'London', 'UK', 8780000),
 (3, 'New York', 'USA', 8500000))

## Problem 8 — Parse Flexible Log Records with Extended Unpacking

A log record is a tuple with this shape:

```python
(timestamp, level, service, *message_parts)
```

The number of message parts can vary.

Write `format_logs(records)` that returns a tuple of formatted strings:

```python
"2026-05-23 10:15:00 | ERROR | billing | payment retry failed"
```

Requirements:

- use extended unpacking
- join `message_parts` with spaces
- reject records with no message parts
- reject records with fewer than 4 fields

In [8]:
def format_logs(records):
    formatted = []

    for index, record in enumerate(records):
        try:
            timestamp, level, service, *message_parts = record
        except ValueError as exc:
            raise ValueError(f"Log record #{index} must contain at least 4 fields") from exc

        if not message_parts:
            raise ValueError(f"Log record #{index} must include a message")

        message = " ".join(str(part) for part in message_parts)
        formatted.append(f"{timestamp} | {level} | {service} | {message}")

    return tuple(formatted)


# Tests
logs = (
    ("2026-05-23 10:15:00", "ERROR", "billing", "payment", "retry", "failed"),
    ("2026-05-23 10:16:00", "INFO", "auth", "user logged in"),
)

expected = (
    "2026-05-23 10:15:00 | ERROR | billing | payment retry failed",
    "2026-05-23 10:16:00 | INFO | auth | user logged in",
)

assert format_logs(logs) == expected

try:
    format_logs([("2026-05-23", "INFO", "auth")])
except ValueError as ex:
    assert "at least 4 fields" in str(ex)
else:
    raise AssertionError("Expected ValueError for incomplete log record")

format_logs(logs)

AssertionError: 

## Problem 9 — Return Multiple Values from a Monte Carlo Simulation

Write `estimate_pi(num_attempts, seed=None)` that returns a tuple:

```python
(pi_estimate, count_inside, num_attempts)
```

Requirements:

- generate random `(x, y)` points in the square `[-1, 1] x [-1, 1]`
- count how many points fall inside the unit circle
- estimate π as `4 * count_inside / num_attempts`
- raise `ValueError` if `num_attempts` is not positive
- support deterministic results with `seed`

In [9]:
from random import Random


def random_shot(rng, radius=1):
    x = rng.uniform(-radius, radius)
    y = rng.uniform(-radius, radius)
    is_inside = x * x + y * y <= radius * radius
    return x, y, is_inside


def estimate_pi(num_attempts, seed=None):
    if not isinstance(num_attempts, int) or num_attempts <= 0:
        raise ValueError("num_attempts must be a positive integer")

    rng = Random(seed)
    count_inside = 0

    for _ in range(num_attempts):
        *_, is_inside = random_shot(rng)
        if is_inside:
            count_inside += 1

    pi_estimate = 4 * count_inside / num_attempts
    return pi_estimate, count_inside, num_attempts


# Tests
pi_estimate, count_inside, attempts = estimate_pi(10_000, seed=42)

assert attempts == 10_000
assert 0 <= count_inside <= attempts
assert 3.0 < pi_estimate < 3.3

try:
    estimate_pi(0)
except ValueError as ex:
    assert "positive integer" in str(ex)
else:
    raise AssertionError("Expected ValueError for non-positive attempts")

pi_estimate, count_inside, attempts

(3.1392, 7848, 10000)

## Problem 10 — Convert Tuple Records to Named Tuples

Raw tuple records are compact, but large codebases often benefit from named fields.

Given city tuples:

```python
(name, country, population)
```

Write `to_city_namedtuples(cities)` that returns a tuple of `City` namedtuple objects.

Then write `largest_city(cities)` that accepts either raw tuples or `City` namedtuples and returns the largest city record.

In [10]:
from collections import namedtuple

City = namedtuple("City", "name country population")


def to_city_namedtuples(cities):
    return tuple(City(name, country, population) for name, country, population in cities)


def largest_city(cities):
    cities = tuple(cities)

    if not cities:
        raise ValueError("cities cannot be empty")

    return max(cities, key=lambda city: city.population if hasattr(city, "population") else city[2])


# Tests
raw_cities = (
    ("London", "UK", 8_780_000),
    ("New York", "USA", 8_500_000),
    ("Beijing", "China", 21_000_000),
)

named_cities = to_city_namedtuples(raw_cities)

assert named_cities[0].name == "London"
assert named_cities[0].country == "UK"
assert named_cities[0].population == 8_780_000

assert largest_city(raw_cities) == ("Beijing", "China", 21_000_000)
assert largest_city(named_cities) == City("Beijing", "China", 21_000_000)

named_cities

(City(name='London', country='UK', population=8780000),
 City(name='New York', country='USA', population=8500000),
 City(name='Beijing', country='China', population=21000000))

## Problem 11 — Build a Tuple-Based Immutable Audit Trail

An audit event is represented as:

```python
(event_id, actor, action, metadata)
```

Where `metadata` is a tuple of key-value pairs:

```python
(("ip", "127.0.0.1"), ("status", "success"))
```

Write `append_event(audit_trail, event)` that:

- validates the event shape
- validates that `metadata` is a tuple of 2-tuples
- returns a new audit trail tuple
- does not mutate the existing audit trail

In [11]:
def validate_audit_event(event):
    try:
        event_id, actor, action, metadata = event
    except ValueError as exc:
        raise ValueError("event must contain exactly 4 fields") from exc

    if not isinstance(event_id, int) or event_id <= 0:
        raise ValueError("event_id must be a positive integer")

    if not isinstance(actor, str) or not actor.strip():
        raise ValueError("actor must be a non-empty string")

    if not isinstance(action, str) or not action.strip():
        raise ValueError("action must be a non-empty string")

    if not isinstance(metadata, tuple):
        raise ValueError("metadata must be a tuple of key-value pairs")

    for pair in metadata:
        try:
            key, value = pair
        except ValueError as exc:
            raise ValueError("each metadata item must be a 2-tuple") from exc

        if not isinstance(key, str) or not key.strip():
            raise ValueError("metadata keys must be non-empty strings")

    return event_id, actor.strip(), action.strip(), metadata


def append_event(audit_trail, event):
    validated_event = validate_audit_event(event)
    return tuple(audit_trail) + (validated_event,)


# Tests
audit_trail = (
    (1, "system", "start", (("status", "success"),)),
)

new_event = (2, "ada", "login", (("ip", "127.0.0.1"), ("status", "success")))
updated_trail = append_event(audit_trail, new_event)

assert len(audit_trail) == 1
assert len(updated_trail) == 2
assert updated_trail[-1] == new_event

try:
    append_event(audit_trail, (3, "ada", "login", ("bad", "metadata")))
except ValueError as ex:
    assert "2-tuple" in str(ex)
else:
    raise AssertionError("Expected ValueError for invalid metadata")

updated_trail

((1, 'system', 'start', (('status', 'success'),)),
 (2, 'ada', 'login', (('ip', '127.0.0.1'), ('status', 'success'))))

## Problem 12 — Design Challenge: Transform Transaction Records

A transaction is represented as:

```python
(transaction_id, account_id, amount, currency, status, *tags)
```

Write `summarize_transactions(transactions)` that returns:

```python
(total_successful_amount, failed_ids, unique_tags)
```

Requirements:

- only transactions with `status == "success"` contribute to the total
- `failed_ids` is a tuple of transaction IDs where `status == "failed"`
- `unique_tags` is a sorted tuple of all unique tags
- use extended unpacking
- validate that every transaction has at least 5 fields

In [12]:
def summarize_transactions(transactions):
    total_successful_amount = 0
    failed_ids = []
    unique_tags = set()

    for index, transaction in enumerate(transactions):
        try:
            transaction_id, account_id, amount, currency, status, *tags = transaction
        except ValueError as exc:
            raise ValueError(
                f"Transaction #{index} must contain at least 5 fields"
            ) from exc

        unique_tags.update(tags)

        if status == "success":
            total_successful_amount += amount
        elif status == "failed":
            failed_ids.append(transaction_id)

    return total_successful_amount, tuple(failed_ids), tuple(sorted(unique_tags))


# Tests
transactions = (
    ("T001", "A100", 125.50, "USD", "success", "card", "online"),
    ("T002", "A200", 90.00, "USD", "failed", "card", "fraud-check"),
    ("T003", "A100", 40.25, "USD", "success", "refund"),
    ("T004", "A300", 10.00, "USD", "pending", "online"),
)

expected = (
    165.75,
    ("T002",),
    ("card", "fraud-check", "online", "refund"),
)

assert summarize_transactions(transactions) == expected

try:
    summarize_transactions([("T005", "A400", 20)])
except ValueError as ex:
    assert "at least 5 fields" in str(ex)
else:
    raise AssertionError("Expected ValueError for short transaction")

summarize_transactions(transactions)

(165.75, ('T002',), ('card', 'fraud-check', 'online', 'refund'))

## Final Reflection Questions

1. When is a tuple better than a list?
2. When is a namedtuple better than a plain tuple?
3. Why can a tuple contain mutable objects even though the tuple itself is immutable?
4. Why is tuple unpacking often more readable than repeated numeric indexing?
5. What assumptions are you making when you treat a tuple as a data record?